In [1]:
library(data.table)
library(tidyverse)
library(parallel)
library(patchwork)
library(pheatmap)
library(ggplot2)
library(ggridges)
library(ape)
library(ggtree)
library(geiger)
library(taxize)

library(asreml)
library(asremlPlus)
library(qqman)
library(topGO)


── Attaching core tidyverse packages ───────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ─────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks da

Online License checked out Fri Apr 25 10:27:57 2025


Loading ASReml-R version 4.2




For example usage please run: vignette('qqman')



Citation appreciated but not required:

Turner, (2018). qqman: an R package for visualizing GWAS results using Q-Q and manhattan plots. Journal of Open Source Software, 3(25), 731, https://doi.org/10.21105/joss.00731.



Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, 

In [2]:
fit_model_v2 <- function(OG_id, data, Kmat,responseVar) {
#   library(asreml)
    gc()
    asreml.options(verbose = FALSE)
    filtered_data <- subset(data, OG == OG_id)
    filtered_data <- filtered_data[,c("assemblyID",responseVar,"PAV","scaled.dn.ds","scaledESM", "scaledPlantCAD", "PMS",
                                      "compdup", "queryCov","pacbio")]
    tmp = filtered_data
    fullFM = as.formula(paste0(responseVar, "~ PAV + PAV:scaled.dn.ds + PAV:scaledESM + PAV:scaledPlantCAD + PAV:PMS + compdup + PAV:queryCov + pacbio"))
    reducedFM = as.formula(paste0(responseVar, "~ compdup + queryCov + pacbio"))

    model_full <- asreml(fixed = fullFM,
                         random = ~ vm(assemblyID, Kmat) , ai.sing = F, data = filtered_data)
    model_reduced1 <- asreml(fixed = fullFM,
                             ai.sing = F, data = filtered_data)
    filtered_data <- tmp
    model_reduced2 <- asreml(fixed = reducedFM,
                             random = ~ vm(assemblyID, Kmat) , ai.sing = F, data = filtered_data)  
    
    totalVar = var(filtered_data[,responseVar])
    modelSummary <- summary(model_full)
    IC_full = infoCriteria(model_full,IClikelihood = "full")
    IC_reduced = infoCriteria(model_reduced2,IClikelihood = "full")
    logLik_full <- IC_full$loglik
    logLik_reduced <- IC_reduced$loglik
    BIC_full <- IC_full$BIC
    BIC_reduced <- IC_reduced$BIC

    LRT_statistic <- -2 * (logLik_reduced - logLik_full)
    p_value <- pchisq(LRT_statistic, df = 5, lower.tail = FALSE)
    gene_explained <-  (model_reduced2$sigma2-model_full$sigma2)/totalVar
    phylo_explained <- (model_reduced1$sigma2-model_full$sigma2)/totalVar
  
    modelWald <- wald.asreml(model_full)
    partialCoeff <- model_full$coefficients$fixed[c(3,8,10,12,16)]
    partialPval <- modelWald[c(2,5:8),4]

#     return(model_full)
    out = c(OG_id,BIC_full,BIC_reduced,LRT_statistic,p_value,gene_explained,phylo_explained,totalVar,partialCoeff,partialPval)
    names(out) = c("OG","BIC_full","BIC_reduced","LR","p","geneVAE","phyloVAE","totalVar",
                   paste("partialCoeff",c("PAV","dNdS","ESM2","plantCad","PMS"),sep = "_"),
                   paste("partialP",c("PAV","dNdS","ESM2","plantCad","PMS"),sep = "_"))
    if(model_full$converge) return(out)
    else return(NULL)
}

In [3]:
metadata = read.delim("/workdir/sh2246/p_phyloGWAS/data/Poaceae_metadata_filtered_2025.04.16.tsv",header = T)

rownames(metadata) = metadata$assemblyID

rhizomeDat = data.table::fread("/workdir/sh2246/p_phyloGWAS/data/grassBase_cleaned_rhizome.txt",header = T,data.table = F)

rhizomeDat = rhizomeDat[,c(2,1102)]

metadata = merge(metadata,rhizomeDat,by = "assemblyID",all.x = T)

metadata$pacbio = as.factor(grepl("pacbio",metadata$technology,ignore.case = T))

phyloKMat <- read.table('/workdir/sh2246/p_phyloGWAS/output/phyloK_728Poaceae_astral_20250407.txt')
colnames(phyloKMat) <- rownames(phyloKMat)
phyloKMat2 <- read.table('/workdir/sh2246/p_phyloGWAS/output/phyloK_728Poaceae_astral_20250416.txt')
colnames(phyloKMat2) <- rownames(phyloKMat2)

In [4]:
envData = read.table("/workdir/sh2246/p_phyloGWAS/output/envData_706Poaceae_20250416.txt")

ePCIdx = grep("PC",colnames(envData))

envDf = cbind(rownames(envData),envData)
colnames(envDf)[1] = "assemblyID"

In [5]:
testData_merged = data.table::fread("/workdir/sh2246/p_phyloGWAS/output/masterDataTable_20250424.txt",header = T,data.table = F,nThread = 20)

testData_merged$PMS = as.factor(testData_merged$PMS)
testData_merged$assemblyID = as.factor(testData_merged$assemblyID)
testData_merged$pacbio = as.factor(testData_merged$pacbio)
testData_merged$groups2 = as.factor(testData_merged$groups2)

In [6]:
# Get all unique assemblyIDs
all_assemblyIDs <- unique(testData_merged$assemblyID)

testData_merged$PAV <- 1

# Expand data so every OG has every assemblyID
expanded_data <- testData_merged %>%
  complete(OG, assemblyID = all_assemblyIDs)

# Fill in missing rows with desired logic
final_data <- expanded_data %>%
  group_by(OG) %>%
  mutate(
    scaled.dn.ds = if_else(is.na(scaled.dn.ds), mean(scaled.dn.ds, na.rm = TRUE), scaled.dn.ds),
    scaledESM = if_else(is.na(scaledESM), mean(scaledESM, na.rm = TRUE), scaledESM),
    scaledPlantCAD = if_else(is.na(scaledPlantCAD), mean(scaledPlantCAD, na.rm = TRUE), scaledPlantCAD),
    queryCov = if_else(is.na(queryCov), mean(queryCov, na.rm = TRUE), queryCov),
    PMS = if_else(is.na(PMS), 0, as.numeric(PMS)),
    PAV = if_else(is.na(PAV), 0, PAV)
  ) %>%
  ungroup()
rm(expand_data)
gc()


# Create a lookup table from testData_merged
metadata_lookup <- testData_merged %>%
  select(assemblyID, compdup, pacbio) %>%
  distinct()

# Fill in only the NA values in final_data using coalesce after joining
final_data <- final_data %>%
  left_join(metadata_lookup, by = "assemblyID", suffix = c("", ".new")) %>%
  mutate(
    compdup = coalesce(compdup, compdup.new),
    pacbio = coalesce(pacbio, pacbio.new)
  ) %>%
  select(-compdup.new, -pacbio.new)

Warning message in rm(expand_data):
“object 'expand_data' not found”


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,22324461,1192.3,47308152,2526.6,47308152,2526.6
Vcells,12970151286,98954.5,19354747241,147665.1,13235086152,100975.7


ERROR: Error in (function (classes, fdef, mtable) : unable to find an inherited method for function ‘select’ for signature ‘"data.frame"’


In [ ]:
testData_merged = final_data
rm(final_data)
rm(metadata_lookup)
gc()
testData_merged$PMS = as.factor(testData_merged$PMS)
testData_merged$assemblyID = as.factor(testData_merged$assemblyID)
testData_merged$pacbio = as.factor(testData_merged$pacbio)
testData_merged$groups2 = as.factor(testData_merged$groups2)

In [ ]:
mappingFile = read.table("/workdir/sh2246/p_phyloGWAS/output/OGToPv_mapping_v2.txt")
mappingFile2 = read.table("/workdir/sh2246/p_phyloGWAS/output/OGToZm_mapping_v2.txt")

colnames(mappingFile) = c("PvID","OG")
colnames(mappingFile2) = c("ZmID","OG")

mappingFileMerged = merge(mappingFile,mappingFile2,by = "OG",all = T)

head(mappingFileMerged)

In [ ]:
asreml.options(maxit = 30,verbose = F)
test = fit_model_v2("OG0021599",data = final_data,phyloKMat,"envPC_5")
test

In [ ]:
asreml.options(maxit = 40,verbose = F)
testRes_envPC1_v2 = mclapply(OG_list,function(x) try(fit_model_v2(x,data = testData_merged,phyloKMat,"envPC_1"),
                                                     silent = T),mc.cores = 30)
testRes_envPC1_v2 = t(simplify2array(testRes_envPC1_v2[sapply(testRes_envPC1_v2,length)==length(test)]))
testRes_envPC1_v2 = as.data.frame(testRes_envPC1_v2)
testRes_envPC1_v2 = merge(testRes_envPC1_v2,mappingFileMerged,by = "OG",all.x = T)
write.table(testRes_envPC1_v2,"/workdir/sh2246/p_phyloGWAS/output/ASREML_res_envPC1_20250425.txt",quote = F,sep = "\t",row.names = F)
testRes_envPC1_v2 = read.table("/workdir/sh2246/p_phyloGWAS/output/ASREML_res_envPC1_20250425.txt",header =T)